*0.1 Python for GenAI · async/await, asyncio.gather, concurrency for API calls*

# concurrency for API calls

**The situation.** The nightly job now uses `gather`. It runs fine on 40 tickets. Tonight there are 2,000. It fires all 2,000 requests in the same instant. The provider allows about 500 a minute. It answers most of them with error **429 — too many requests**, the job crashes, and nothing is indexed.

**The problem is not speed, it is bursts.** Sending everything at once is like 2,000 people walking into a shop with one door. What you need is a door that lets a few through at a time, and a clock that limits how many enter per second.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

import asyncio
from concurrent.futures import ThreadPoolExecutor


# A notebook already has an event loop running, so asyncio.run() is not allowed here.
# This helper runs the async code on a separate thread instead. In a normal script you
# would simply write asyncio.run(main()).
def run_async(coroutine):
    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, coroutine).result()

**Two guards, one for each.** A *semaphore* is a counter: "at most 5 requests in flight at once." A *rate limiter* is a clock: "at most 10 requests may start per second." Each request must pass both before it goes out.

In [2]:
import time

from aiolimiter import AsyncLimiter
from openai import AsyncOpenAI

tickets = []
for number in range(1, 61):
    tickets.append(f"Ticket {number}: customer cannot log in after password reset")

MAX_IN_FLIGHT = 5  # the door: how many requests may be open at once
REQUESTS_PER_SECOND = 10  # the clock: how many may start per second


async def embed_one(ai: AsyncOpenAI, ticket: str) -> int:
    reply = await ai.embeddings.create(model="text-embedding-3-small", input=ticket)
    return len(reply.data[0].embedding)


async def guarded_batch() -> int:
    door = asyncio.Semaphore(MAX_IN_FLIGHT)
    clock = AsyncLimiter(REQUESTS_PER_SECOND, time_period=1)
    async with AsyncOpenAI(timeout=30) as ai:

        async def guarded(ticket: str) -> int:
            async with door, clock:  # wait for a free slot AND a free tick, then go
                return await embed_one(ai, ticket)

        tasks = []
        for ticket in tickets:
            tasks.append(guarded(ticket))
        results = await asyncio.gather(
            *tasks
        )  # still gather — but each request waits at the guards
        return len(results)


started = time.perf_counter()
count = run_async(guarded_batch())
guarded_seconds = time.perf_counter() - started
print(count, "tickets with guards:", round(guarded_seconds, 1), "s, zero errors")
assert guarded_seconds >= len(tickets) / REQUESTS_PER_SECOND * 0.8

60 tickets with guards: 5.3 s, zero errors


**Reading the number.** 60 tickets at 10 per second is about 6 seconds — exactly what the clock allows. The job is slower than an unguarded burst, and that is the point: it finishes, every night, without a single 429.

```
no guards      2,000 requests ──▶ provider ──▶ 429 429 429 … job fails
with guards    2,000 requests ──▶ [5 at a time] ──▶ [10 per second] ──▶ provider ──▶ done in ≈ 200 s
```

**Where the numbers come from.** The provider tells you your limits in every reply, in headers named `x-ratelimit-limit-requests` and `x-ratelimit-limit-tokens`. Read them once and set the two guards from them — do not guess.

In [3]:
from openai import OpenAI

# One small request, and the reply headers that state the account's limits.
raw = OpenAI(timeout=30).chat.completions.with_raw_response.create(
    model=MODEL, messages=[{"role": "user", "content": "hi"}], max_tokens=1
)
for name in [
    "x-ratelimit-limit-requests",
    "x-ratelimit-remaining-requests",
    "x-ratelimit-limit-tokens",
]:
    print(f"{name:<32} {raw.headers.get(name)}")
assert raw.headers.get("x-ratelimit-limit-requests") is not None

x-ratelimit-limit-requests       10000
x-ratelimit-remaining-requests   9998
x-ratelimit-limit-tokens         200000


**Reading the headers.** `limit-requests` is how many requests per minute the account may make; `limit-tokens` is the token budget per minute. Divide by 60 for a per-second clock, and by the number of machines running the job.

| Use it when | Don't when | Instead use |
|---|---|---|
| any job or service that sends many requests to a provider — always | never skip it; only the two numbers change per provider | a gateway (LiteLLM, an API gateway) that enforces limits for all your services in one place; the provider's batch API for overnight work |

**Watch out**
- Limits are per account, not per program. 4 machines × 10 per second = 40 per second. Divide the limit by the number of machines.
- Providers also limit *tokens* per minute. A few very long texts can exhaust that before the request limit does.
- Even with guards, a real 429 can arrive. Keep retries with backoff (a later item) as the safety net.